# 12. 재현, ablation, 통계적 비교

논문 한 편을 만들려면 `내 방법이 좋아 보인다`가 아니라 `무엇 때문에 좋아졌는지`를 보여야 합니다.
이 노트북은 실험 조건별 결과를 만들고 bootstrap 신뢰구간을 계산합니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
rng = np.random.default_rng(42)
n = 200
conditions = rng.choice(["clean", "noise", "blur", "alarm"], size=n, p=[0.35, 0.25, 0.25, 0.15])

def simulate_correct(system, cond):
    base = {"clean": 0.72, "noise": 0.52, "blur": 0.48, "alarm": 0.55}[cond]
    bonus = {
        "ocr_only": 0.00,
        "ocr_layout": 0.07 if cond in ["blur", "clean"] else 0.03,
        "asr_ocr": 0.10 if cond in ["noise", "clean"] else 0.05,
        "asr_ocr_sound_gate": 0.11 if cond != "blur" else 0.05,
    }[system]
    return rng.random() < min(0.95, base + bonus)

systems = ["ocr_only", "ocr_layout", "asr_ocr", "asr_ocr_sound_gate"]
rows = []
for i, cond in enumerate(conditions):
    for sys in systems:
        rows.append({"sample_id": i, "condition": cond, "system": sys, "correct": simulate_correct(sys, cond)})
res = pd.DataFrame(rows)
pivot = res.groupby(["system", "condition"])["correct"].mean().reset_index()
pivot.to_csv(ARTIFACTS / "ablation_by_condition.csv", index=False, encoding="utf-8-sig")
pivot


In [ ]:
def bootstrap_ci(values, n_boot=2000, alpha=0.05):
    vals = np.asarray(values).astype(float)
    boots = [np.mean(np.random.choice(vals, size=len(vals), replace=True)) for _ in range(n_boot)]
    return float(np.mean(vals)), float(np.quantile(boots, alpha / 2)), float(np.quantile(boots, 1 - alpha / 2))

ci_rows = []
for sys in systems:
    vals = res[res.system == sys]["correct"]
    mean, lo, hi = bootstrap_ci(vals)
    ci_rows.append({"system": sys, "accuracy": mean, "ci_low": lo, "ci_high": hi})
ci = pd.DataFrame(ci_rows).sort_values("accuracy", ascending=False)
ci.to_csv(ARTIFACTS / "ablation_bootstrap_ci.csv", index=False, encoding="utf-8-sig")
ci


In [ ]:
if plt:
    fig, ax = plt.subplots(figsize=(8, 4))
    order = ci.sort_values("accuracy")["system"]
    plot_df = ci.set_index("system").loc[order]
    ax.barh(plot_df.index, plot_df["accuracy"], xerr=[
        plot_df["accuracy"] - plot_df["ci_low"],
        plot_df["ci_high"] - plot_df["accuracy"],
    ])
    ax.set_xlabel("Accuracy")
    ax.set_title("Ablation with bootstrap CI")
    fig.tight_layout()
    fig.savefig(ARTIFACTS / "ablation_ci.png", dpi=160)
